!rm -rf AMRBART
!git clone https://github.com/Phucgiacat/AMRBART-VietNamese.git AMRBART
%cd AMRBART


## 1️⃣ Kiểm tra GPU & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyvi phonlp
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1mX2r_TV0avyyDO-v10UmllLcetZbM96T?usp=sharing -O /content/data
print("\n✅ Đã tải xong dữ liệu thô vào /content/AMRBART/template/data!")

Retrieving folder contents
Processing file 1kV-mOu-cd5s6ULJmnsxetXtSndAygrXZ dev.amr
Processing file 18s49gvTheBRWX5prAs4Od2fPO3hU1Nll test.amr
Processing file 1wT-zMtAJhwevFhHpMpw9TuFCF76msP0w train.amr
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1kV-mOu-cd5s6ULJmnsxetXtSndAygrXZ
To: /content/data_v2/dev.amr
100% 832k/832k [00:00<00:00, 142MB/s]
Downloading...
From: https://drive.google.com/uc?id=18s49gvTheBRWX5prAs4Od2fPO3hU1Nll
To: /content/data_v2/test.amr
100% 855k/855k [00:00<00:00, 154MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wT-zMtAJhwevFhHpMpw9TuFCF76msP0w
To: /content/data_v2/train.amr
100% 16.5M/16.5M [00:00<00:00, 48.3MB/s]
Download completed

✅ Đã tải xong dữ liệu thô vào /content/AMRBART/template/data!


In [ ]:
import os

# ================================================================
# ⚙️  CẤU HÌNH – SỬA NẾU CẦN
# ================================================================
DRIVE_DATA_DIR   = '/content/data_v2'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/outputs_v6'

# Đường dẫn local cho models (tải bằng system Python để tránh lỗi Hub)
MODEL_LOCAL     = '/content/AMRBART-large-v2'    # xfbai/AMRBART-large-v2
TOKENIZER_LOCAL = '/content/facebook-bart-large'  # facebook/bart-large

DATASET   = 'ViAMR'
LR        = '1e-5'
TRAIN_BSZ = 4       # T4=4, A100=16
EVAL_BSZ  = 4
GRAD_ACC  = 4       # effective batch = TRAIN_BSZ * GRAD_ACC
NUM_EPOCHS  = 10
EARLY_STOP  = 10
WARMUP      = 200
NUM_BEAMS   = 5

# Paths (tự động)
REPO_DIR     = '/content/AMRBART'
VENV_DIR     = f'{REPO_DIR}/.venv'
PYTHON       = f'{VENV_DIR}/bin/python'
FINETUNE_DIR = f'{REPO_DIR}/fine-tune'
DATA_DIR     = f'{FINETUNE_DIR}/data/{DATASET}'
CACHE_DIR    = f'{REPO_DIR}/.cache/model'
DATA_CACHE   = f'{DATA_DIR}/.cache/dump-amrparsing'
OUTPUT_DIR   = f'{DRIVE_OUTPUT_DIR}/{DATASET}-AMRParsing-bsz{TRAIN_BSZ*GRAD_ACC}-lr{LR}'
TRAIN_SH     = f'{FINETUNE_DIR}/train-AMRBART-large-AMRParsing.sh'

for d in [DRIVE_OUTPUT_DIR, DATA_DIR, CACHE_DIR, DATA_CACHE, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('📋 Cấu hình:')
print(f'  Model local     : {MODEL_LOCAL}')
print(f'  Tokenizer local : {TOKENIZER_LOCAL}')
print(f'  Data dir        : {DATA_DIR}')
print(f'  Output dir      : {OUTPUT_DIR}')
print(f'  Effective BSZ   : {TRAIN_BSZ}×{GRAD_ACC} = {TRAIN_BSZ*GRAD_ACC}')

print('\n📂 Data files:')
for f in ['train.amr', 'dev.amr', 'test.amr']:
    p = os.path.join(DRIVE_DATA_DIR, f)
    if os.path.exists(p):
        print(f'  ✅ {f}: {os.path.getsize(p)/1024:.1f} KB')
    else:
        print(f'  ❌ {f}: KHÔNG TÌM THẤY!')

📋 Cấu hình:
  Model local     : /content/AMRBART-large-v2
  Tokenizer local : /content/facebook-bart-large
  Data dir        : /content/AMRBART/fine-tune/data/ViAMR
  Output dir      : /content/drive/MyDrive/outputs_v6/ViAMR-AMRParsing-bsz16-lr1e-5
  Effective BSZ   : 4×4 = 16

📂 Data files:
  ✅ train.amr: 16124.2 KB
  ✅ dev.amr: 812.0 KB
  ✅ test.amr: 835.4 KB


## 2️⃣ Clone AMRBART & Setup Python 3.8 với `uv`

In [ ]:
!git clone https://github.com/Phucgiacat/AMRBART-VietNamese.git AMRBART
%cd AMRBART

Cloning into '/content/AMRBART'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 400 (delta 43), reused 28 (delta 27), pack-reused 333 (from 1)
Receiving objects: 100% (400/400), 6.79 MiB | 34.57 MiB/s, done.
Resolving deltas: 100% (186/186), done.
base_trainer.py			  inference-text.sh
common				  main.py
data_interface			  metric
Eval-AMRBART-large-AMR2Text.sh	  model_interface
Eval-AMRBART-large-AMRParsing.sh  seq2seq_trainer.py
evaluation			  train-AMRBART-large-AMR2Text.sh
inference-amr.sh		  train-AMRBART-large-AMRParsing.sh


In [ ]:
!pip install -q uv
print('✅ uv installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 99.0 MB/s eta 0:00:00
✅ uv installed


In [ ]:
%%bash
uv venv /content/AMRBART/.venv --python 3.8
echo "✅ venv: $(/content/AMRBART/.venv/bin/python --version)"

✅ venv: Python 3.8.20


Using CPython 3.8.20
Creating virtual environment at: AMRBART/.venv
Activate with: source AMRBART/.venv/bin/activate


In [ ]:

%%bash
VENV=/content/AMRBART/.venv

uv pip install --python "$VENV" \
    'transformers==4.21.3' \
    'datasets==2.4.0' \
    torch \
    'penman>=1.1.0' \
    smatch \
    nltk \
    sacrebleu \
    rouge-score \
    networkx \
    regex \
    cached_property \
    filelock \
    tensorboard \
    'pyvi' \
    'phonlp' \
    tqdm

$VENV/bin/python -c "import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)"

echo ''
$VENV/bin/python -c "import transformers; print('✅ transformers:', transformers.__version__)"
$VENV/bin/python -c "import tokenizers;   print('✅ tokenizers  :', tokenizers.__version__)"
$VENV/bin/python -c "import penman;       print('✅ penman      :', penman.__version__)"
$VENV/bin/python -c "import torch;        print('✅ torch       :', torch.__version__)"


✅ transformers: 4.21.3
✅ tokenizers  : 0.12.1
✅ penman      : 1.3.1
✅ torch       : 2.4.1+cu121


Using Python 3.8.20 environment at: AMRBART/.venv
Resolved 88 packages in 2.55s
Prepared 84 packages in 50.46s
Installed 88 packages in 222ms
 + absl-py==2.3.1
 + aiohappyeyeballs==2.4.4
 + aiohttp==3.10.11
 + aiosignal==1.3.1
 + async-timeout==5.0.1
 + attrs==25.3.0
 + cached-property==2.0.1
 + certifi==2026.5.20
 + cffi==1.17.1
 + charset-normalizer==3.4.7
 + click==8.1.8
 + colorama==0.4.6
 + cryptography==47.0.0
 + datasets==2.4.0
 + dill==0.3.5.1
 + filelock==3.16.1
 + frozenlist==1.5.0
 + fsspec==2025.3.0
 + google-auth==2.50.0
 + google-auth-oauthlib==1.0.0
 + grpcio==1.70.0
 + hf-xet==1.5.1
 + huggingface-hub==0.36.2
 + idna==3.15
 + importlib-metadata==8.5.0
 + jinja2==3.1.6
 + joblib==1.4.2
 + lxml==6.1.1
 + markdown==3.7
 + markupsafe==2.1.5
 + mpmath==1.3.0
 + multidict==6.1.0
 + multiprocess==0.70.13
 + networkx==3.1
 + nltk==3.9.1
 + numpy==1.24.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime

## 3️⃣ Download Models về local

> **Tại sao download local?**  
> `transformers==4.21.3` trong venv Python 3.8 dùng `huggingface_hub` cũ → không tương thích với Hub API mới → lỗi `MissingSchema` và `OSError: Can't load tokenizer`.  
> **Giải pháp**: dùng **system Python** để tải cả model lẫn tokenizer về local, rồi truyền đường dẫn local vào script.

In [ ]:
from huggingface_hub import snapshot_download

# ── 1. Download xfbai/AMRBART-large-v2 ───────────────────────────────────
print(f'[1/2] Downloading xfbai/AMRBART-large-v2 → {MODEL_LOCAL} ...')
snapshot_download(
    repo_id='xfbai/AMRBART-large-v2',
    local_dir=MODEL_LOCAL,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print(f'✅ Model OK')
!ls "{MODEL_LOCAL}"

# ── 2. Download facebook/bart-large (tokenizer files) ────────────────────
print(f'\n[2/2] Downloading facebook/bart-large tokenizer → {TOKENIZER_LOCAL} ...')
snapshot_download(
    repo_id='facebook/bart-large',
    local_dir=TOKENIZER_LOCAL,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', '*.bin', 'pytorch_model*', 'rust_model.ot'],
)
print(f'✅ Tokenizer OK')
!ls "{TOKENIZER_LOCAL}"

[1/2] Downloading xfbai/AMRBART-large-v2 → /content/AMRBART-large-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Model OK
config.json  model.safetensors	README.md		 tokenizer_config.json
merges.txt   pytorch_model.bin	special_tokens_map.json  vocab.json

[2/2] Downloading facebook/bart-large tokenizer → /content/facebook-bart-large ...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Tokenizer OK
config.json  README.md		    tokenizer.json
merges.txt   tokenizer_config.json  vocab.json


In [ ]:
import json

# Tắt use_cache trong config → hết warning khi dùng gradient_checkpointing
cfg_path = f'{MODEL_LOCAL}/config.json'
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['use_cache'] = False
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print('✅ use_cache=False patched in config.json')


✅ use_cache=False patched in config.json


## 4️⃣ Preprocessing: `.amr` → `.jsonl`

In [ ]:
%%writefile /content/preprocess_vi_amr.py
#!/usr/bin/env python3
"""Convert Vietnamese AMR .amr files to AMRBART-compatible .jsonl"""
import re, os, copy, json, penman, argparse
from tqdm import tqdm
from penman.models.noop import NoOpModel

noop_model = NoOpModel()


def _tokenize_encoded_graph(encoded):
    linearized = re.sub(r'(".+?")', r' \1 ', encoded)
    pieces = []
    for piece in linearized.split():
        if piece.startswith('"') and piece.endswith('"'):
            pieces.append(piece)
        else:
            piece = (piece.replace('(', ' ( ').replace(')', ' ) ')
                         .replace(':', ' :').replace('/', ' / ').strip())
            pieces.append(piece)
    return re.sub(r'\s+', ' ', ' '.join(pieces)).strip().split(' ')


def dfs_linearize(graph):
    g = copy.deepcopy(graph)
    g.metadata = {}
    nodes = _tokenize_encoded_graph(penman.encode(g, model=noop_model))
    remap = {}
    for i in range(1, len(nodes)):
        if nodes[i] == '/':
            remap[nodes[i - 1]] = f'<pointer:{len(remap)}>'
    i, result = 1, [nodes[0]]
    while i < len(nodes):
        nxt, lst = nodes[i], result[-1]
        if nxt in remap:
            if lst == '(' and i + 1 < len(nodes) and nodes[i + 1] == '/':
                nxt = remap[nxt]
                i += 1
            elif lst.startswith(':'):
                nxt = remap[nxt]
        result.append(nxt)
        i += 1
    return result


def process_file(src, out_prefix):
    print(f'\nProcessing: {src}')
    graphs = penman.load(str(src), model=noop_model)
    print(f'  Loaded {len(graphs)} graphs')
    sents, amrs, errs = [], [], 0
    for i, g in enumerate(tqdm(graphs, desc='  Linearizing')):
        try:
            snt = g.metadata.get('tok', '').strip()
            if not snt:
                errs += 1
                continue
            amrs.append(' '.join(dfs_linearize(g)))
            sents.append(snt)
        except Exception as e:
            print(f'  skip [{i}]: {e}')
            errs += 1
    print(f'  OK={len(sents)}, Errors={errs}')
    with open(out_prefix + '.jsonl', 'w', encoding='utf-8') as f:
        for s, a in zip(sents, amrs):
            f.write(json.dumps({'sent': s, 'amr': a}, ensure_ascii=False) + '\n')
    print(f'  Written → {out_prefix}.jsonl')
    return len(sents)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--input_dir',  required=True)
    ap.add_argument('--output_dir', required=True)
    args = ap.parse_args()
    os.makedirs(args.output_dir, exist_ok=True)
    total = 0
    for split, fname in [('train', 'train.amr'), ('val', 'dev.amr'), ('test', 'test.amr')]:
        src = os.path.join(args.input_dir, fname)
        if not os.path.exists(src):
            print(f'SKIP (not found): {src}')
            continue
        total += process_file(src, os.path.join(args.output_dir, split))
    print(f'\n✅ Done! Total {total} examples.')

Writing /content/preprocess_vi_amr.py


In [ ]:
!{PYTHON} /content/preprocess_vi_amr.py \
    --input_dir  "{DRIVE_DATA_DIR}" \
    --output_dir "{DATA_DIR}"


Processing: /content/data_v2/train.amr
  Loaded 18519 graphs

  OK=18519, Errors=0
  Written → /content/AMRBART/fine-tune/data/ViAMR/train.jsonl

Processing: /content/data_v2/dev.amr
  Loaded 1137 graphs

  OK=1137, Errors=0
  Written → /content/AMRBART/fine-tune/data/ViAMR/val.jsonl

Processing: /content/data_v2/test.amr
  Loaded 1137 graphs

  OK=1137, Errors=0
  Written → /content/AMRBART/fine-tune/data/ViAMR/test.jsonl

✅ Done! Total 20793 examples.


In [ ]:
import json, os

print('=== Sample train.jsonl ===')
with open(f'{DATA_DIR}/train.jsonl', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        d = json.loads(line)
        print(f"\n[{i+1}] sent: {d['sent'][:90]}")
        print(f"      amr : {d['amr'][:90]}")

print('\n=== Số mẫu ===')
for sp in ['train', 'val', 'test']:
    p = f'{DATA_DIR}/{sp}.jsonl'
    if os.path.exists(p):
        n = sum(1 for _ in open(p, encoding='utf-8'))
        print(f'  {sp:6s}: {n} mẫu')

=== Sample train.jsonl ===

[1] sent: đó là tình_thương con của người mẹ .
      amr : ( <pointer:0> tình :domain ( <pointer:1> đó ) :compound ( <pointer:2> thương :patient ( <p

[2] sent: Nó gợi_ý rằng chúng_ta quan_tâm tới sự đấu_tranh , tới thách_thức .
      amr : ( <pointer:0> gợi_ý :ARG0 ( <pointer:1> nó ) :ARG1 ( <pointer:2> quan_tâm :ARG0 ( <pointer

=== Số mẫu ===
  train : 18519 mẫu
  val   : 1137 mẫu
  test  : 1137 mẫu


## 5️⃣ Gold AMR files cho Smatch Evaluation

In [ ]:
import shutil, os

for src_name, dst_name in [('dev.amr', 'val-gold.amr'), ('test.amr', 'test-gold.amr')]:
    src = os.path.join(DRIVE_DATA_DIR, src_name)
    dst = os.path.join(DATA_DIR, dst_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ {src_name} → {dst_name}')
    else:
        print(f'❌ Không thấy: {src}')

✅ dev.amr → val-gold.amr
✅ test.amr → test-gold.amr


## 6️⃣ Patch Shell Script

In [ ]:
import shutil
shutil.copy2(TRAIN_SH, TRAIN_SH + '.bak')
print(f'✅ Backup: {TRAIN_SH}.bak')

✅ Backup: /content/AMRBART/fine-tune/train-AMRBART-large-AMRParsing.sh.bak


In [ ]:
# ── 1. Paths ──────────────────────────────────────────────────────────────
!sed -i 's|^BasePath=.*|BasePath={REPO_DIR}|g'       "{TRAIN_SH}"
!sed -i 's|^Dataset=LDC2020|Dataset={DATASET}|g'     "{TRAIN_SH}"
!sed -i 's|^DataPath=.*|DataPath={DATA_DIR}|g'       "{TRAIN_SH}"
!sed -i 's|^ModelCache=.*|ModelCache={CACHE_DIR}|g'  "{TRAIN_SH}"
!sed -i 's|^DataCache=.*|DataCache={DATA_CACHE}|g'   "{TRAIN_SH}"
print('✅ Paths patched')

# ── 2. Output dir → Google Drive ─────────────────────────────────────────
!sed -i 's|^OutputDir=.*|OutputDir={OUTPUT_DIR}|g'   "{TRAIN_SH}"
# Xoá interactive prompt "already exists?" (Colab không interactive)
!sed -i '/read -p/,/esac/c\  rm -rf ${{OutputDir}}; mkdir -p ${{OutputDir}}' "{TRAIN_SH}"
print('✅ OutputDir patched')

# ── 3. Tokenizer: dùng local path thay vì facebook/bart-large ────────────
!sed -i 's|--tokenizer_name "facebook/bart-large"|--tokenizer_name "{TOKENIZER_LOCAL}"|g' "{TRAIN_SH}"
print('✅ Tokenizer_name patched → {TOKENIZER_LOCAL}')

# ── 4. Hyperparameters ───────────────────────────────────────────────────
!sed -i 's|^lr=.*|lr={LR}|g'  "{TRAIN_SH}"
!sed -i 's|--per_device_train_batch_size [0-9]*|--per_device_train_batch_size {TRAIN_BSZ}|g'   "{TRAIN_SH}"
!sed -i 's|--per_device_eval_batch_size [0-9]*|--per_device_eval_batch_size {EVAL_BSZ}|g'     "{TRAIN_SH}"
!sed -i 's|--gradient_accumulation_steps [0-9]*|--gradient_accumulation_steps {GRAD_ACC}|g'  "{TRAIN_SH}"
!sed -i 's|--num_train_epochs [0-9]*|--num_train_epochs {NUM_EPOCHS}|g'                       "{TRAIN_SH}"
!sed -i 's|--early_stopping [0-9]*|--early_stopping {EARLY_STOP}|g'                           "{TRAIN_SH}"
!sed -i 's|--warmup_steps [0-9]*|--warmup_steps {WARMUP}|g'                                   "{TRAIN_SH}"
!sed -i 's|--generation_num_beams [0-9]*|--generation_num_beams {NUM_BEAMS}|g'                "{TRAIN_SH}"
print('✅ Hyperparameters patched')

# ── 5. Workers ───────────────────────────────────────────────────────────
!sed -i 's|--dataloader_num_workers [0-9]*|--dataloader_num_workers 2|g'           "{TRAIN_SH}"
!sed -i 's|--eval_dataloader_num_workers [0-9]*|--eval_dataloader_num_workers 1|g' "{TRAIN_SH}"
print('✅ Workers patched')

# ── 6. Gradient checkpointing (Python str.replace — tránh sed multiline) ─
with open(TRAIN_SH, 'r') as fh:
    content = fh.read()
content = content.replace(
    '--fp16_backend "auto" \\',
    '--fp16_backend "auto" \\\n    --gradient_checkpointing \\'
)
with open(TRAIN_SH, 'w') as fh:
    fh.write(content)
print('✅ Gradient checkpointing added')

print('\n=== Script sau khi patch ===')
!cat "{TRAIN_SH}"

✅ Paths patched
✅ OutputDir patched
✅ Tokenizer_name patched → {TOKENIZER_LOCAL}
✅ Hyperparameters patched
✅ Workers patched
✅ Gradient checkpointing added

=== Script sau khi patch ===
export CUDA_VISIBLE_DEVICES=0
RootDir="$( cd "$( dirname "${BASH_SOURCE[0]}" )" >/dev/null 2>&1 && pwd )"

Dataset=ViAMR
#Dataset=LDC2017

BasePath=/content/AMRBART
DataPath=/content/AMRBART/fine-tune/data/ViAMR

ModelCate=AMRBART-large

MODEL=$1
ModelCache=/content/AMRBART/.cache/model
DataCache=/content/AMRBART/fine-tune/data/ViAMR/.cache/dump-amrparsing

lr=1e-5

OutputDir=/content/drive/MyDrive/outputs_v6/ViAMR-AMRParsing-bsz16-lr1e-5

if [ ! -d ${OutputDir} ];then
  mkdir -p ${OutputDir}
else
  rm -rf ${OutputDir}; mkdir -p ${OutputDir}
fi

export HF_DATASETS_CACHE=$DataCache

if [ ! -d ${DataCache} ];then
  mkdir -p ${DataCache}
fi

# torchrun --nnodes=1 --nproc_per_node=1 --max_restarts=0 --rdzv_id=1 --rdzv_backend=c10d main.py \
python -u main.py \
    --data_dir $DataPath \
    --task "text2amr

## 7️⃣ Chạy Training

In [ ]:
"""
🩹 Vá bug trong source code AMRBART gốc (GitHub)
Chạy cell này SAU bước clone repo và TRƯỚC bước Training.

Sửa 5 bug:
  1. Loss Dilution: mean() chia cho tổng token (cả padding=0) → gradient pha loãng
     → Fix: chia cho số token thật (non-padding) → loss nhất quán mọi batch size
  2. --max_grad_norm 0 → 1.0 (bật gradient clipping)
  3. str(n) trong tokenization_bart.py (fix crash int+str)
  4. calculate_smatch tổng hợp đúng F-score (utils.py)
  5. Mở khóa trainer.save_model() → xuất pytorch_model.bin (main.py)
"""
import os

FINETUNE_DIR = '/content/AMRBART/fine-tune'
MAIN_PY  = f'{FINETUNE_DIR}/main.py'
TOK_PY   = f'{FINETUNE_DIR}/model_interface/tokenization_bart.py'
UTIL_PY  = f'{FINETUNE_DIR}/common/utils.py'
SEQ2SEQ  = f'{FINETUNE_DIR}/seq2seq_trainer.py'
TRAIN_SH = f'{FINETUNE_DIR}/train-AMRBART-large-AMRParsing.sh'

patched = []

# ═══════════════════════════════════════════════════════════════════════════
# Bug 1 (QUAN TRỌNG): Loss Dilution trong label_smoothed_nll_loss
# ═══════════════════════════════════════════════════════════════════════════
# Code gốc dùng mean() chia cho TỔNG SỐ token (bao gồm padding đã bị zero).
# Khi batch lớn → padding chiếm 70%+ → loss bị pha loãng → gradient cực nhỏ
# → model không học được → sinh rác () → Smatch = 0
# Fix: chia cho số token THẬT (non-padding)
with open(SEQ2SEQ, 'r') as f:
    code = f.read()

old_loss = (
    '    if ignore_index is not None:\n'
    '        pad_mask = target.eq(ignore_index)\n'
    '        nll_loss.masked_fill_(pad_mask, 0.0)\n'
    '        smooth_loss.masked_fill_(pad_mask, 0.0)\n'
    '    else:\n'
    '        nll_loss = nll_loss.squeeze(-1)\n'
    '        smooth_loss = smooth_loss.squeeze(-1)\n'
    '\n'
    '    # nll_loss = nll_loss.sum()                   # mean()? Scared to break other math.\n'
    '    # smooth_loss = smooth_loss.sum()\n'
    '    nll_loss = nll_loss.mean()  # mean()? Scared to break other math.\n'
    '    smooth_loss = smooth_loss.mean()'
)

new_loss = (
    '    if ignore_index is not None:\n'
    '        pad_mask = target.eq(ignore_index)\n'
    '        nll_loss.masked_fill_(pad_mask, 0.0)\n'
    '        smooth_loss.masked_fill_(pad_mask, 0.0)\n'
    '        # Chia cho số token THẬT (non-padding) thay vì tổng số token\n'
    '        count = (~pad_mask).sum().clamp(min=1)\n'
    '        nll_loss = nll_loss.sum() / count\n'
    '        smooth_loss = smooth_loss.sum() / count\n'
    '    else:\n'
    '        nll_loss = nll_loss.squeeze(-1)\n'
    '        smooth_loss = smooth_loss.squeeze(-1)\n'
    '        nll_loss = nll_loss.mean()\n'
    '        smooth_loss = smooth_loss.mean()'
)

if old_loss in code:
    code = code.replace(old_loss, new_loss)
    with open(SEQ2SEQ, 'w') as f:
        f.write(code)
    patched.append('seq2seq_trainer.py: loss normalization fix (non-padding count)')
else:
    print('ℹ️  seq2seq_trainer.py: loss normalization đã OK hoặc pattern khác')

# ═══════════════════════════════════════════════════════════════════════════
# Bug 2: --max_grad_norm 0 → 1.0 (trong shell script)
# ═══════════════════════════════════════════════════════════════════════════
if os.path.exists(TRAIN_SH):
    with open(TRAIN_SH, 'r') as f:
        sh = f.read()
    if '--max_grad_norm 0' in sh:
        sh = sh.replace('--max_grad_norm 0', '--max_grad_norm 1.0')
        with open(TRAIN_SH, 'w') as f:
            f.write(sh)
        patched.append('shell: --max_grad_norm 1.0')
    else:
        print('ℹ️  shell: max_grad_norm đã OK')

# ═══════════════════════════════════════════════════════════════════════════
# Bug 3: str(n) trong tokenization_bart.py
# ═══════════════════════════════════════════════════════════════════════════
with open(TOK_PY, 'r') as f:
    code = f.read()

old_node = '            else:\n                nodes_.append(n)\n        nodes = nodes_'
new_node = '            else:\n                nodes_.append(str(n))\n        nodes = nodes_'

if old_node in code:
    code = code.replace(old_node, new_node)
    with open(TOK_PY, 'w') as f:
        f.write(code)
    patched.append('tokenization_bart.py: str(n) fix')
else:
    print('ℹ️  tokenization_bart.py: str(n) đã OK')

# ═══════════════════════════════════════════════════════════════════════════
# Bug 4: calculate_smatch tổng hợp đúng F-score
# ═══════════════════════════════════════════════════════════════════════════
with open(UTIL_PY, 'r') as f:
    code = f.read()

if 'pass   # consume all pairs' in code:
    old_smatch = (
        '        with Path(predictions_path).open() as p, Path(test_path).open() as g:\n'
        '            for score in smatch.score_amr_pairs(p, g):\n'
        '                pass   # consume all pairs; score holds cumulative P/R/F\n'
        '        if score is None:\n'
        '            print("[calculate_smatch] WARNING: no AMR pairs were scored (empty files?)")\n'
        '            return {"smatch": 0.0}\n'
        '        return {"smatch": score[2]}'
    )
    new_smatch = (
        '        match_tot, test_tot, gold_tot = 0, 0, 0\n'
        '        with Path(predictions_path).open() as p, Path(test_path).open() as g:\n'
        '            for match_num, test_num, gold_num in smatch.score_amr_pairs(p, g):\n'
        '                match_tot += match_num\n'
        '                test_tot += test_num\n'
        '                gold_tot += gold_num\n'
        '        if test_tot == 0 or gold_tot == 0:\n'
        '            return {"smatch": 0.0}\n'
        '        pr = float(match_tot) / float(test_tot)\n'
        '        rc = float(match_tot) / float(gold_tot)\n'
        '        return {"smatch": 2 * (pr * rc) / (pr + rc) if (pr + rc) > 0 else 0.0}'
    )
    code = code.replace(old_smatch, new_smatch)
    with open(UTIL_PY, 'w') as f:
        f.write(code)
    patched.append('utils.py: smatch aggregation fix')
else:
    print('ℹ️  utils.py: smatch đã OK')

# ═══════════════════════════════════════════════════════════════════════════
# Bug 5: Mở khóa trainer.save_model()
# ═══════════════════════════════════════════════════════════════════════════
with open(MAIN_PY, 'r') as f:
    code = f.read()

if '        # trainer.save_model()' in code:
    code = code.replace(
        '        # trainer.save_model()  # Saves the tokenizer too for easy upload',
        '        trainer.save_model()  # Saves the tokenizer too for easy upload'
    )
    with open(MAIN_PY, 'w') as f:
        f.write(code)
    patched.append('main.py: trainer.save_model() enabled')
else:
    print('ℹ️  main.py: trainer.save_model() đã OK')

# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
if patched:
    for p in patched:
        print(f'  ✅ {p}')
    print('=' * 60)
    print('🎉 Tất cả bug đã được vá! Sẵn sàng train.')
else:
    print('  ℹ️  Không có gì cần vá (đã vá hết rồi)')
    print('=' * 60)


ℹ️  utils.py: smatch đã OK

  ✅ seq2seq_trainer.py: loss normalization fix (non-padding count)
  ✅ shell: --max_grad_norm 1.0
  ✅ tokenization_bart.py: str(n) fix
  ✅ main.py: trainer.save_model() enabled
🎉 Tất cả bug đã được vá! Sẵn sàng train.


In [ ]:
%%bash
cd /content/AMRBART/fine-tune
echo "=== PREPARING DATA WITH PhoNLP ==="
python3 prepare_vietnamese_amr.py


In [ ]:
%%bash
source /content/AMRBART/.venv/bin/activate

echo "=== Python   : $(python --version) ==="
echo "=== torch    : $(python -c 'import torch; print(torch.__version__)') ==="
echo "=== GPU CUDA : $(python -c 'import torch; print(torch.cuda.is_available())') ==="
echo "======================================="

cd /content/AMRBART/fine-tune
# Truyền đường dẫn LOCAL cho cả model và tokenizer
bash train-AMRBART-large-AMRParsing.sh /content/AMRBART-large-v2

Process is terminated.


## 9️⃣ (Optional) Inference – Parse câu tiếng Việt mới

In [ ]:
import json

# sentences đã có từ trước
# sentences = [s["sent"] for s in list_snt]

list_snt = []
with open("/content/AMRBART/fine-tune/data/ViAMR/val.jsonl", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        list_snt.append(data)

sentences = [s["sent"] for s in list_snt]

output_path = "/content/AMRBART/fine-tune/data/ViAMR/data4parsing.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for sent in sentences:
        obj = {
            "sent": sent,
            "amr": ""
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
%%bash
SCRIPT_PATH="/content/AMRBART/fine-tune/inference-amr.sh"
sed -i 's|^BasePath=.*|BasePath=/content/AMRBART|g' $SCRIPT_PATH
sed -i '/read -p/,/esac/c\  rm -rf ${OutputDir}; mkdir -p ${OutputDir}' $SCRIPT_PATH
bash inference-amr.sh /content/drive/MyDrive/outputs_v6/ViAMR-AMRParsing-bsz16-lr1e-5/checkpoint-xxx  # SỬA LẠI SỐ CHECKPOINT PHÙ HỢP SAU KHI TRAIN


In [ ]:
%%bash
source /content/AMRBART/.venv/bin/activate

echo "=== Python   : $(python --version) ==="
echo "=== torch    : $(python -c 'import torch; print(torch.__version__)') ==="
echo "=== GPU CUDA : $(python -c 'import torch; print(torch.cuda.is_available())') ==="
echo "======================================="

cd /content/AMRBART/fine-tune
# Truyền đường dẫn LOCAL cho cả model và tokenizer
bash inference-amr.sh /content/drive/MyDrive/outputs_v6/ViAMR-AMRParsing-bsz16-lr1e-5/checkpoint-xxx  # SỬA LẠI SỐ CHECKPOINT PHÙ HỢP SAU KHI TRAIN


=== Python   : Python 3.8.20 ===
=== torch    : 2.4.1+cu121 ===
=== GPU CUDA : True ===
06/17/2026 08:14:36 - WARNING - __main__ - Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: True
06/17/2026 08:14:36 - INFO - __main__ - Training/evaluation parameters Seq2SeqTrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_pin_memory=True,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=False,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=True,
do_train=False,
early_stopping=5,
eval_accumulation_steps=None,
eval_dataloader_num_workers=2,
eval_delay=0,
eval_lenpen=1.0,
eval_steps=None,
evaluation_strategy=no,
fp16=True,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_min_num_params=0,
full_determinism=False,
generat

## 🎯 Hướng dẫn Inference
Đoạn code dưới đây sẽ nạp lại các tham số đã train để sinh ra đồ thị AMR từ câu tiếng Việt.

In [ ]:
# 🚀 Chạy Inference với mô hình đã train (AMRParsing)
import torch
from transformers import AutoConfig
from model_interface.modeling_bart import BartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
import pyvi.ViTokenizer as ViTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "./outputs/LDC2020-AMRBART-large-AMRParing-bsz16-lr-1e-5-UnifiedInp" # Cập nhật đúng đường dẫn checkpoint

print("Loading model from:", model_path)
tokenizer = AMRBartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path).to(device)

def parse_vietnamese_text(text):
    # Tiền xử lý: Tách từ bằng Pyvi
    segmented_text = ViTokenizer.tokenize(text)
    print("Segmented Input:", segmented_text)
    
    input_ids = tokenizer(segmented_text, return_tensors="pt")["input_ids"]
    unified_input = torch.cat([
        torch.tensor([[tokenizer.bos_token_id, tokenizer.mask_token_id, tokenizer.eos_token_id]]),
        input_ids[:, 1:-1], 
        torch.tensor([[tokenizer.amr_eos_token_id]])
    ], dim=-1).to(device)
    
    outputs = model.generate(
        unified_input, 
        max_length=1024, 
        num_beams=5, 
        decoder_start_token_id=tokenizer.amr_bos_token_id
    )
    
    amr_tokens = tokenizer.convert_ids_to_tokens(outputs[0])
    amr_tokens = [t for t in amr_tokens if t not in ['<s>', '</s>', '<pad>', '<amr>']]
    return " ".join(amr_tokens).replace("Ġ", " ")

test_sentence = "Lúc ấy, có quan tể tướng là Tiêu Vũ bước ra."
print("Kết quả AMR:")
print(parse_vietnamese_text(test_sentence))
